#### 1489. Find Critical and Pseudo-Critical Edges in Minimum Spanning Tree

* https://leetcode.com/problems/find-critical-and-pseudo-critical-edges-in-minimum-spanning-tree/description/


In [ ]:
class UnionFind:
    def __init__(self, size):
        self.parent = list(range(size))
        self.rank = [0]*size
        self.components = size
    
    def find(self, node):
        if self.parent[node] != node:
            self.parent[node] = self.find(self.parent[node])
        return self.parent[node]
    
    def union(self, nodea, nodeb):
        roota = self.find(nodea)
        rootb = self.find(nodeb)

        if roota == rootb:
            return False
        
        if self.rank[roota] < self.rank[rootb]:
            self.parent[roota] = rootb
        elif self.rank[roota] > self.rank[rootb]:
            self.parent[rootb] = roota
        else:
            self.parent[rootb] = roota
            self.rank[roota] += 1
        
        self.components -= 1
        return True

class Solution:
    def findCriticalAndPseudoCriticalEdges(self, n: int, edges: List[List[int]]) -> List[List[int]]:
        """
        Returns:
        [
            critical_edges_indices,
            pseudo_critical_edges_indices
        ]
        """

        # Step 1: Attach original index to each edge
        indexed_edges: List[Tuple[int, int, int, int]] = [
            (u, v, weight, idx)
            for idx, (u, v, weight) in enumerate(edges)
        ]

        # Step 2: Sort edges by weight (Kruskal requirement)
        indexed_edges.sort(key=lambda edge: edge[2])

        def build_mst(
            exclude_edge_idx: int = -1,
            include_edge_idx: int = -1
        ) -> int:
            """
            Builds MST with optional:
            - excluding one edge
            - force-including one edge

            Returns:
                MST total weight OR float('inf') if not spanning
            """
            uf = UnionFind(n)
            total_weight = 0

            # Step A: Force include edge if required
            if include_edge_idx != -1:
                u, v, w, _ = indexed_edges[include_edge_idx]
                if uf.union(u, v):
                    total_weight += w

            # Step B: Standard Kruskal
            for i, (u, v, w, _) in enumerate(indexed_edges):
                if i == exclude_edge_idx:
                    continue

                if uf.union(u, v):
                    total_weight += w

                # Early stop: MST complete
                if uf.components == 1:
                    break

            # If not fully connected → invalid MST
            return total_weight if uf.components == 1 else float("inf")

        # Step 3: Compute baseline MST cost
        baseline_mst_cost = build_mst()

        critical_edges = []
        pseudo_critical_edges = []

        # Step 4: Classify each edge
        for i in range(len(indexed_edges)):
            # Case 1: Exclude edge → check if critical
            cost_without_edge = build_mst(exclude_edge_idx=i)

            if cost_without_edge > baseline_mst_cost:
                # Edge is critical
                critical_edges.append(indexed_edges[i][3])
                continue

            # Case 2: Force include → check pseudo-critical
            cost_with_edge = build_mst(include_edge_idx=i)

            if cost_with_edge == baseline_mst_cost:
                pseudo_critical_edges.append(indexed_edges[i][3])

        return [critical_edges, pseudo_critical_edges]
        